<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 3 - Ednalvo - Evaluation of Compositions

## Setup

In [1]:
from pathlib import Path
import os


def find_phase_dir(start: Path | None = None) -> Path:
    """
    Find the Phase 3 project directory in a portable way.

    Priority:
    1. Environment variable CL_ST1_PH3_DIR, if defined.
    2. Current working directory or one of its parents.
    3. A child directory named cl_st1_ph3_ednalvo under the current directory or parents.
    """
    env_dir = os.environ.get("CL_ST1_PH3_DIR")
    if env_dir:
        phase_dir = Path(env_dir).expanduser().resolve()
        if phase_dir.exists():
            return phase_dir
        raise FileNotFoundError(f"CL_ST1_PH3_DIR does not exist: {phase_dir}")

    start = (start or Path.cwd()).resolve()
    search_roots = [start, *start.parents]

    candidates = []
    for root in search_roots:
        candidates.append(root)
        candidates.append(root / "cl_st1_ph3_ednalvo")

    for candidate in candidates:
        if (candidate / "corpus").is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the Phase 3 directory. "
        "Run the notebook from the project tree or set CL_ST1_PH3_DIR."
    )


PHASE_DIR = find_phase_dir()

CORPUS_DIR = PHASE_DIR / "corpus"
FONTES_DIR = CORPUS_DIR / "00_fontes"
COMPOSICOES_DIR = CORPUS_DIR / "01_composicoes"
RESUMOS_DIR = CORPUS_DIR / "02_resumos"
ANOTADAS_DIR = CORPUS_DIR / "03_composicoes_anotadas"
AVALIACAO_DIR = PHASE_DIR / "avaliacao_composicoes"

print(f"Using Phase 3 directory: {PHASE_DIR}")

Using Phase 3 directory: /home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo


In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_DIR = PHASE_DIR

ORIGINAL_SCORES_PATH = PROJECT_DIR / "corpus" / "00_fontes" / "composicoes_de_admissao_universitaria.tsv"
AI_ASSESSMENTS_DIR = PROJECT_DIR / "corpus" / "04_composicoes_avaliadas"
SAS_DIR = PROJECT_DIR / "sas"

SCORE_COLS = [
    "adequacao_ao_tema",
    "adequacao_a_coletanea",
    "adequacao_ao_tipo_de_texto",
    "adequacao_a_norma_padrao",
    "coesao",
    "coerencia",
]

TOTAL_COL = "pontuacao_total"

CANDIDATE_GROUP_ORDER = [
    "humano_maiores_notas",
    "humano_menores_notas",
    "gemini_menores_notas_espelho",
    "gpt_menores_notas_espelho",
]

In [3]:
for path in [PROJECT_DIR, ORIGINAL_SCORES_PATH, AI_ASSESSMENTS_DIR, SAS_DIR]:
    print(path, "OK" if path.exists() else "MISSING")

/home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo OK
/home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo/corpus/00_fontes/composicoes_de_admissao_universitaria.tsv OK
/home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo/corpus/04_composicoes_avaliadas OK
/home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo/sas OK


## Load and standardise human-assessed original scores

In [4]:
df_human_scores = pd.read_csv(ORIGINAL_SCORES_PATH, sep="\t")


def portable_path_string(path_value: str | Path, base_dir: Path = PROJECT_DIR) -> str:
    path = Path(path_value).expanduser()
    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.resolve().relative_to(base_dir.resolve()).as_posix()
    except ValueError:
        return path.as_posix()


df_human_scores["base_id"] = df_human_scores["inscricao"].astype(str)

df_human_scores["candidate_source"] = "humano"
df_human_scores["candidate_band"] = df_human_scores["grupo"].map({
    "250_maiores_notas": "maiores_notas",
    "250_menores_notas": "menores_notas",
})

df_human_scores["candidate_group"] = df_human_scores["candidate_band"].map({
    "maiores_notas": "humano_maiores_notas",
    "menores_notas": "humano_menores_notas",
})

df_human_scores["is_mirror"] = False
df_human_scores["mirror_model"] = pd.NA

df_human_scores["assessor_source"] = "humano"
df_human_scores["assessor_model"] = pd.NA
df_human_scores["score_source_file"] = portable_path_string(ORIGINAL_SCORES_PATH)

df_human_scores["composition_id"] = (
        df_human_scores["base_id"] + "_" + df_human_scores["candidate_group"]
)

df_human_scores = df_human_scores.rename(columns={
    "arquivo_original": "filename_original",
    "caminho_em_composicoes": "path",
})

df_human_scores["path"] = df_human_scores["path"].map(
    lambda value: portable_path_string(value) if pd.notna(value) else value
)

for col in SCORE_COLS + [TOTAL_COL]:
    df_human_scores[col] = pd.to_numeric(df_human_scores[col], errors="coerce")

df_human_scores["score_sum_check"] = df_human_scores[SCORE_COLS].sum(axis=1)
df_human_scores["score_residual"] = (
        df_human_scores[TOTAL_COL] - df_human_scores["score_sum_check"]
)

df_human_scores.head()

,grupo,inscricao,filename_original,path,anotacoes,adequacao_ao_tema,adequacao_a_coletanea,adequacao_ao_tipo_de_texto,adequacao_a_norma_padrao,coesao,...,candidate_band,candidate_group,is_mirror,mirror_model,assessor_source,assessor_model,score_source_file,composition_id,score_sum_check,score_residual
0,250_maiores_notas,36237,36237 (0519-029).txt,corpus/01_composicoes/maiores_notas/36237.txt,{FO: inc findas},4.5,5.0,5.0,5.0,5.0,...,maiores_notas,humano_maiores_notas,False,<NA>,humano,<NA>,corpus/00_fontes/composicoes_de_admissao_unive...,36237_humano_maiores_notas,29.5,0.0
1,250_maiores_notas,46656,46656 (0414-009).txt,corpus/01_composicoes/maiores_notas/46656.txt,NaN,5.0,5.0,4.0,4.5,5.0,...,maiores_notas,humano_maiores_notas,False,<NA>,humano,<NA>,corpus/00_fontes/composicoes_de_admissao_unive...,46656_humano_maiores_notas,28.5,0.0
2,250_maiores_notas,36560,36560 (0502-031).txt,corpus/01_composicoes/maiores_notas/36560.txt,{FO: poluição}; {FO: más},4.5,4.5,4.5,5.0,5.0,...,maiores_notas,humano_maiores_notas,False,<NA>,humano,<NA>,corpus/00_fontes/composicoes_de_admissao_unive...,36560_humano_maiores_notas,28.0,0.0
3,250_maiores_notas,36923,36923 (0416-024).txt,corpus/01_composicoes/maiores_notas/36923.txt,NaN,5.0,4.0,4.5,5.0,5.0,...,maiores_notas,humano_maiores_notas,False,<NA>,humano,<NA>,corpus/00_fontes/composicoes_de_admissao_unive...,36923_humano_maiores_notas,28.0,0.0
4,250_maiores_notas,45355,45355 (0477-030).txt,corpus/01_composicoes/maiores_notas/45355.txt,"(terra, fogo, ar e água); (terra, fogo, ar e á...",4.5,4.5,4.5,5.0,5.0,...,maiores_notas,humano_maiores_notas,False,<NA>,humano,<NA>,corpus/00_fontes/composicoes_de_admissao_unive...,45355_humano_maiores_notas,28.0,0.0


## Parse AI-assessment Markdown files

In [5]:
CRITERION_PATTERNS = {
    "adequacao_ao_tema": r"Adequa[cç][aã]o ao Tema",
    "adequacao_a_coletanea": r"Adequa[cç][aã]o [àa] Colet[âa]nea",
    "adequacao_ao_tipo_de_texto": r"Adequa[cç][aã]o ao tipo de texto",
    "adequacao_a_norma_padrao": r"Adequa[cç][aã]o [àa] norma padr[aã]o",
    "coesao": r"Coes[aã]o",
    "coerencia": r"Coer[êe]ncia",
    "pontuacao_total": r"Pontua[cç][aã]o Total",
}


def parse_ptbr_number(value: str) -> float:
    value = value.strip()
    value = value.replace("**", "")
    value = value.replace(",", ".")
    value = re.sub(r"[^0-9.\-]", "", value)
    return float(value) if value else np.nan


def extract_score_from_markdown(text: str, criterion_regex: str) -> float:
    pattern = rf"\|\s*[^|]*{criterion_regex}[^|]*\|\s*([^|]+?)\s*\|"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        return np.nan
    return parse_ptbr_number(match.group(1))


def parse_ai_assessment_file(path: Path) -> dict:
    text = path.read_text(encoding="utf-8")

    base_id_match = re.search(r"(\d+)_avaliacao_", path.name)
    base_id = base_id_match.group(1) if base_id_match else pd.NA

    row = {
        "base_id": str(base_id),
        "ai_assessment_filename": path.name,
        "ai_assessment_path": portable_path_string(path),
        "score_source_file": portable_path_string(path),
    }

    for col, pattern in CRITERION_PATTERNS.items():
        row[col] = extract_score_from_markdown(text, pattern)

    return row

Quickly test on one file:

In [6]:
example_file = next((AI_ASSESSMENTS_DIR / "maiores_notas").glob("*_avaliacao_*.md"))
parse_ai_assessment_file(example_file)

{'base_id': '37662',
 'ai_assessment_filename': '37662_avaliacao_gpt-5.6-sol.md',
 'ai_assessment_path': 'corpus/04_composicoes_avaliadas/maiores_notas/37662_avaliacao_gpt-5.6-sol.md',
 'score_source_file': 'corpus/04_composicoes_avaliadas/maiores_notas/37662_avaliacao_gpt-5.6-sol.md',
 'adequacao_ao_tema': 4.0,
 'adequacao_a_coletanea': 4.0,
 'adequacao_ao_tipo_de_texto': 4.0,
 'adequacao_a_norma_padrao': 3.0,
 'coesao': 3.0,
 'coerencia': 4.0,
 'pontuacao_total': 22.0}

## Load all AI-assessed scores

In [7]:
AI_GROUP_MAP = {
    "maiores_notas": {
        "candidate_group": "humano_maiores_notas",
        "candidate_source": "humano",
        "candidate_band": "maiores_notas",
        "is_mirror": False,
        "mirror_model": pd.NA,
    },
    "menores_notas": {
        "candidate_group": "humano_menores_notas",
        "candidate_source": "humano",
        "candidate_band": "menores_notas",
        "is_mirror": False,
        "mirror_model": pd.NA,
    },
    "menores_notas_gemini": {
        "candidate_group": "gemini_menores_notas_espelho",
        "candidate_source": "gemini",
        "candidate_band": "menores_notas_espelho",
        "is_mirror": True,
        "mirror_model": "gemini",
    },
    "menores_notas_gpt": {
        "candidate_group": "gpt_menores_notas_espelho",
        "candidate_source": "gpt",
        "candidate_band": "menores_notas_espelho",
        "is_mirror": True,
        "mirror_model": "gpt",
    },
}


def load_ai_assessment_folder(folder_name: str, assessor_model: str = "gpt-5.6-sol") -> pd.DataFrame:
    if folder_name not in AI_GROUP_MAP:
        raise ValueError(f"Unknown AI assessment folder: {folder_name}")

    folder = AI_ASSESSMENTS_DIR / folder_name
    files = sorted(folder.glob("*_avaliacao_*.md"))

    if not files:
        raise FileNotFoundError(f"No AI assessment files found in: {folder}")

    rows = [parse_ai_assessment_file(path) for path in files]
    df = pd.DataFrame(rows)

    meta = AI_GROUP_MAP[folder_name]
    for key, value in meta.items():
        df[key] = value

    df["assessor_source"] = "gpt_ai"
    df["assessor_model"] = assessor_model

    df["composition_id"] = df["base_id"] + "_" + df["candidate_group"]
    df["filename"] = df["base_id"] + ".txt"

    for col in SCORE_COLS + [TOTAL_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["score_sum_check"] = df[SCORE_COLS].sum(axis=1)
    df["score_residual"] = df[TOTAL_COL] - df["score_sum_check"]

    return df


df_ai_scores = pd.concat(
    [
        load_ai_assessment_folder("maiores_notas"),
        load_ai_assessment_folder("menores_notas"),
        load_ai_assessment_folder("menores_notas_gemini"),
        load_ai_assessment_folder("menores_notas_gpt"),
    ],
    ignore_index=True,
)

df_ai_scores.head()

,base_id,ai_assessment_filename,ai_assessment_path,score_source_file,adequacao_ao_tema,adequacao_a_coletanea,adequacao_ao_tipo_de_texto,adequacao_a_norma_padrao,coesao,coerencia,...,candidate_source,candidate_band,is_mirror,mirror_model,assessor_source,assessor_model,composition_id,filename,score_sum_check,score_residual
0,32757,32757_avaliacao_gpt-5.6-sol.md,corpus/04_composicoes_avaliadas/maiores_notas/...,corpus/04_composicoes_avaliadas/maiores_notas/...,2.5,2.5,3.5,3.0,3.5,4.0,...,humano,maiores_notas,False,<NA>,gpt_ai,gpt-5.6-sol,32757_humano_maiores_notas,32757.txt,19.0,0.0
1,32875,32875_avaliacao_gpt-5.6-sol.md,corpus/04_composicoes_avaliadas/maiores_notas/...,corpus/04_composicoes_avaliadas/maiores_notas/...,3.0,3.0,5.0,2.0,4.0,5.0,...,humano,maiores_notas,False,<NA>,gpt_ai,gpt-5.6-sol,32875_humano_maiores_notas,32875.txt,22.0,0.0
2,33083,33083_avaliacao_gpt-5.6-sol.md,corpus/04_composicoes_avaliadas/maiores_notas/...,corpus/04_composicoes_avaliadas/maiores_notas/...,2.5,2.5,3.5,4.5,3.5,3.5,...,humano,maiores_notas,False,<NA>,gpt_ai,gpt-5.6-sol,33083_humano_maiores_notas,33083.txt,20.0,0.0
3,33228,33228_avaliacao_gpt-5.6-sol.md,corpus/04_composicoes_avaliadas/maiores_notas/...,corpus/04_composicoes_avaliadas/maiores_notas/...,3.0,3.0,4.0,0.0,4.0,4.0,...,humano,maiores_notas,False,<NA>,gpt_ai,gpt-5.6-sol,33228_humano_maiores_notas,33228.txt,18.0,0.0
4,33383,33383_avaliacao_gpt-5.6-sol.md,corpus/04_composicoes_avaliadas/maiores_notas/...,corpus/04_composicoes_avaliadas/maiores_notas/...,3.0,3.0,4.0,2.0,3.5,4.0,...,humano,maiores_notas,False,<NA>,gpt_ai,gpt-5.6-sol,33383_humano_maiores_notas,33383.txt,19.5,0.0


## Build the canonical long-format score table

In [8]:
COMMON_SCORE_COLS = [
    "composition_id",
    "base_id",
    "candidate_source",
    "candidate_group",
    "candidate_band",
    "is_mirror",
    "mirror_model",
    "assessor_source",
    "assessor_model",
    "path",
    "filename",
    "filename_original",
    "ai_assessment_filename",
    "ai_assessment_path",
    "score_source_file",
    *SCORE_COLS,
    TOTAL_COL,
    "score_sum_check",
    "score_residual",
]

df_human_scores_aligned = df_human_scores.reindex(columns=COMMON_SCORE_COLS)
df_ai_scores_aligned = df_ai_scores.reindex(columns=COMMON_SCORE_COLS)

df_scores_long = pd.concat(
    [
        df_human_scores_aligned,
        df_ai_scores_aligned,
    ],
    ignore_index=True,
)

path_metadata_cols = ["path", "ai_assessment_path", "score_source_file"]

for col in path_metadata_cols:
    df_scores_long[col] = df_scores_long[col].map(
        lambda value: portable_path_string(Path(value))
        if pd.notna(value) and str(value).strip()
        else pd.NA
    )

for col in SCORE_COLS + [TOTAL_COL, "score_sum_check", "score_residual"]:
    df_scores_long[col] = pd.to_numeric(df_scores_long[col], errors="coerce")

unknown_candidate_groups = sorted(
    set(df_scores_long["candidate_group"].dropna()) - set(CANDIDATE_GROUP_ORDER)
)

if unknown_candidate_groups:
    raise ValueError(f"Unknown candidate groups: {unknown_candidate_groups}")

df_scores_long["candidate_group"] = pd.Categorical(
    df_scores_long["candidate_group"],
    categories=CANDIDATE_GROUP_ORDER,
    ordered=True,
)

df_scores_long.head()

,composition_id,base_id,candidate_source,candidate_group,candidate_band,is_mirror,mirror_model,assessor_source,assessor_model,path,...,score_source_file,adequacao_ao_tema,adequacao_a_coletanea,adequacao_ao_tipo_de_texto,adequacao_a_norma_padrao,coesao,coerencia,pontuacao_total,score_sum_check,score_residual
0,36237_humano_maiores_notas,36237,humano,humano_maiores_notas,maiores_notas,False,<NA>,humano,<NA>,corpus/01_composicoes/maiores_notas/36237.txt,...,corpus/00_fontes/composicoes_de_admissao_unive...,4.5,5.0,5.0,5.0,5.0,5.0,29.5,29.5,0.0
1,46656_humano_maiores_notas,46656,humano,humano_maiores_notas,maiores_notas,False,<NA>,humano,<NA>,corpus/01_composicoes/maiores_notas/46656.txt,...,corpus/00_fontes/composicoes_de_admissao_unive...,5.0,5.0,4.0,4.5,5.0,5.0,28.5,28.5,0.0
2,36560_humano_maiores_notas,36560,humano,humano_maiores_notas,maiores_notas,False,<NA>,humano,<NA>,corpus/01_composicoes/maiores_notas/36560.txt,...,corpus/00_fontes/composicoes_de_admissao_unive...,4.5,4.5,4.5,5.0,5.0,4.5,28.0,28.0,0.0
3,36923_humano_maiores_notas,36923,humano,humano_maiores_notas,maiores_notas,False,<NA>,humano,<NA>,corpus/01_composicoes/maiores_notas/36923.txt,...,corpus/00_fontes/composicoes_de_admissao_unive...,5.0,4.0,4.5,5.0,5.0,4.5,28.0,28.0,0.0
4,45355_humano_maiores_notas,45355,humano,humano_maiores_notas,maiores_notas,False,<NA>,humano,<NA>,corpus/01_composicoes/maiores_notas/45355.txt,...,corpus/00_fontes/composicoes_de_admissao_unive...,4.5,4.5,4.5,5.0,5.0,4.5,28.0,28.0,0.0


Conceptually:
- `df_scores_long` has one row per composition × assessor.
- The original human essays should have human and AI rows.
- The LLM mirrored essays should have AI rows only.

## Validation checks

In [9]:
print("Rows by candidate group and assessor:")
display(
    df_scores_long
    .groupby(["candidate_group", "assessor_source"], observed=False)
    .size()
    .reset_index(name="n")
)

print("Unique compositions by candidate group:")
display(
    df_scores_long[["composition_id", "candidate_group"]]
    .drop_duplicates()
    .groupby("candidate_group", observed=False)
    .size()
    .reset_index(name="n_compositions")
)

print("Score residual summary:")
display(df_scores_long["score_residual"].describe())

print("Missing AI score fields:")
display(df_ai_scores[SCORE_COLS + [TOTAL_COL]].isna().sum())

print("Duplicate assessment rows:")
duplicate_mask = df_scores_long.duplicated(
    subset=["composition_id", "assessor_source", "assessor_model"],
    keep=False,
)

duplicate_count = duplicate_mask.sum()
display(duplicate_count)

if duplicate_count:
    display(
        df_scores_long.loc[
            duplicate_mask,
            [
                "composition_id",
                "base_id",
                "candidate_group",
                "assessor_source",
                "assessor_model",
                "score_source_file",
                TOTAL_COL,
            ],
        ].sort_values(
            ["composition_id", "assessor_source", "assessor_model"]
        )
    )

Rows by candidate group and assessor:


,candidate_group,assessor_source,n
0,humano_maiores_notas,gpt_ai,250
1,humano_maiores_notas,humano,250
2,humano_menores_notas,gpt_ai,250
3,humano_menores_notas,humano,250
4,gemini_menores_notas_espelho,gpt_ai,250
5,gemini_menores_notas_espelho,humano,0
6,gpt_menores_notas_espelho,gpt_ai,250
7,gpt_menores_notas_espelho,humano,0


Unique compositions by candidate group:


,candidate_group,n_compositions
0,humano_maiores_notas,250
1,humano_menores_notas,250
2,gemini_menores_notas_espelho,250
3,gpt_menores_notas_espelho,250


Score residual summary:


count    1500.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: score_residual, dtype: float64

Missing AI score fields:


adequacao_ao_tema             0
adequacao_a_coletanea         0
adequacao_ao_tipo_de_texto    0
adequacao_a_norma_padrao      0
coesao                        0
coerencia                     0
pontuacao_total               0
dtype: int64

Duplicate assessment rows:


np.int64(0)

## Human vs AI assessor comparison

In [10]:
df_assessor_compare = (
    df_scores_long
    .query("candidate_source == 'humano'")
    .pivot_table(
        index=["composition_id", "base_id", "candidate_band", "candidate_group"],
        columns="assessor_source",
        values=[*SCORE_COLS, TOTAL_COL],
        aggfunc="first",
    )
)

df_assessor_compare.columns = [
    f"{score}_{assessor}"
    for score, assessor in df_assessor_compare.columns
]

df_assessor_compare = df_assessor_compare.reset_index()

df_assessor_compare["total_diff_ai_minus_humano"] = (
        df_assessor_compare["pontuacao_total_gpt_ai"]
        - df_assessor_compare["pontuacao_total_humano"]
)

df_assessor_compare["total_abs_diff"] = (
    df_assessor_compare["total_diff_ai_minus_humano"].abs()
)

df_assessor_compare.head()

,composition_id,base_id,candidate_band,candidate_group,adequacao_a_coletanea_gpt_ai,adequacao_a_coletanea_humano,adequacao_a_norma_padrao_gpt_ai,adequacao_a_norma_padrao_humano,adequacao_ao_tema_gpt_ai,adequacao_ao_tema_humano,adequacao_ao_tipo_de_texto_gpt_ai,adequacao_ao_tipo_de_texto_humano,coerencia_gpt_ai,coerencia_humano,coesao_gpt_ai,coesao_humano,pontuacao_total_gpt_ai,pontuacao_total_humano,total_diff_ai_minus_humano,total_abs_diff
0,32610_humano_menores_notas,32610,menores_notas,humano_menores_notas,2.0,1.0,0.0,2.5,3.0,1.0,3.5,2.0,3.0,1.5,2.0,3.5,13.5,11.5,2.0,2.0
1,32674_humano_menores_notas,32674,menores_notas,humano_menores_notas,2.0,1.0,0.0,0.5,3.0,1.0,3.0,2.0,2.0,1.0,2.0,1.0,12.0,6.5,5.5,5.5
2,32714_humano_menores_notas,32714,menores_notas,humano_menores_notas,2.5,1.0,1.0,1.0,3.5,1.0,3.0,2.5,3.0,1.5,2.0,2.0,15.0,9.0,6.0,6.0
3,32733_humano_menores_notas,32733,menores_notas,humano_menores_notas,2.0,2.0,0.0,1.0,3.0,1.5,3.0,2.5,3.0,2.0,3.0,2.0,14.0,11.0,3.0,3.0
4,32749_humano_menores_notas,32749,menores_notas,humano_menores_notas,1.0,1.0,4.0,1.5,3.0,1.0,3.0,2.5,3.0,2.0,3.0,3.5,17.0,11.5,5.5,5.5


## Human low originals vs LLM rewrites

In [11]:
df_candidate_compare = (
    df_scores_long
    .query("assessor_source == 'gpt_ai'")
    .pivot_table(
        index="base_id",
        columns="candidate_group",
        values=[*SCORE_COLS, TOTAL_COL],
        aggfunc="first",
    )
)

df_candidate_compare.columns = [
    f"{score}_{group}"
    for score, group in df_candidate_compare.columns
]

df_candidate_compare = df_candidate_compare.reset_index()

df_candidate_compare["gain_gemini_vs_humano_menores_notas"] = (
        df_candidate_compare["pontuacao_total_gemini_menores_notas_espelho"]
        - df_candidate_compare["pontuacao_total_humano_menores_notas"]
)

df_candidate_compare["gain_gpt_vs_humano_menores_notas"] = (
        df_candidate_compare["pontuacao_total_gpt_menores_notas_espelho"]
        - df_candidate_compare["pontuacao_total_humano_menores_notas"]
)

df_candidate_compare["diff_gpt_minus_gemini"] = (
        df_candidate_compare["pontuacao_total_gpt_menores_notas_espelho"]
        - df_candidate_compare["pontuacao_total_gemini_menores_notas_espelho"]
)

df_candidate_compare.head()

,base_id,adequacao_a_coletanea_humano_maiores_notas,adequacao_a_coletanea_humano_menores_notas,adequacao_a_coletanea_gemini_menores_notas_espelho,adequacao_a_coletanea_gpt_menores_notas_espelho,adequacao_a_norma_padrao_humano_maiores_notas,adequacao_a_norma_padrao_humano_menores_notas,adequacao_a_norma_padrao_gemini_menores_notas_espelho,adequacao_a_norma_padrao_gpt_menores_notas_espelho,adequacao_ao_tema_humano_maiores_notas,...,coesao_humano_menores_notas,coesao_gemini_menores_notas_espelho,coesao_gpt_menores_notas_espelho,pontuacao_total_humano_maiores_notas,pontuacao_total_humano_menores_notas,pontuacao_total_gemini_menores_notas_espelho,pontuacao_total_gpt_menores_notas_espelho,gain_gemini_vs_humano_menores_notas,gain_gpt_vs_humano_menores_notas,diff_gpt_minus_gemini
0,32610,NaN,2.0,4.5,4.0,NaN,0.0,5.0,5.0,NaN,...,2.0,5.0,5.0,NaN,13.5,29.0,29.0,15.5,15.5,0.0
1,32674,NaN,2.0,4.0,4.0,NaN,0.0,5.0,5.0,NaN,...,2.0,5.0,5.0,NaN,12.0,27.0,29.0,15.0,17.0,2.0
2,32714,NaN,2.5,4.0,4.0,NaN,1.0,5.0,5.0,NaN,...,2.0,5.0,5.0,NaN,15.0,27.5,29.0,12.5,14.0,1.5
3,32733,NaN,2.0,4.0,5.0,NaN,0.0,5.0,5.0,NaN,...,3.0,5.0,5.0,NaN,14.0,28.5,30.0,14.5,16.0,1.5
4,32749,NaN,1.0,4.5,4.0,NaN,4.0,5.0,5.0,NaN,...,3.0,5.0,5.0,NaN,17.0,29.5,29.0,12.5,12.0,-0.5


## Export assessment tables

In [12]:
OUTPUT_DIR = PROJECT_DIR / "avaliacao_composicoes"
OUTPUT_DIR.mkdir(exist_ok=True)

outputs = {
    "scores_long_dataset": df_scores_long,
    "comparacao_de_avaliadores": df_assessor_compare,
    "comparacao_de_candidatos": df_candidate_compare,
}

for name, df in outputs.items():
    df.to_json(
        OUTPUT_DIR / f"{name}.ndjson",
        orient="records",
        lines=True,
        force_ascii=False,
    )
    df.to_csv(
        OUTPUT_DIR / f"{name}.tsv",
        sep="\t",
        index=False,
    )
    df.to_excel(
        OUTPUT_DIR / f"{name}.xlsx",
        index=False,
    )

print(f"Saved assessment outputs to: {OUTPUT_DIR}")

Saved assessment outputs to: /home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo/avaliacao_composicoes
